# Integracja z Kafką

----

## Subskrypcja tematu

- wymaga uruchomienia Kafki i utworzenia tematu `topic1` w sposób wskazany w pliku `kafka_commands`

Odpowiednik podania parametru --packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.6 w wywołaniu spark-submit

In [ ]:
from os import environ
environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.6 pyspark-shell'

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f

In [ ]:
spark = SparkSession.builder.appName("Structured").config("spark.sql.shuffle.partitions", 4).getOrCreate()

In [ ]:
raw = spark \
.readStream \
.format("kafka") \
.option("kafka.bootstrap.servers", "kafka:9092") \
.option("subscribe", "topic1") \
.load()

In [ ]:
words = raw.select(f.explode(f.split(raw.value, " ")).alias("word"))
wordCounts = words.groupBy("word").count()

In [ ]:
query = wordCounts.writeStream.outputMode("complete").format("console").start()
query.awaitTermination(60)
query.stop()

----

## Wysyłanie wiadomości do tematu

- wymaga uruchomienia Kafki i utworzenia tematów `topic1` i `topic2` w sposób wskazany w pliku `kafka_commands`

In [ ]:
from os import environ
environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.6 pyspark-shell'

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f

In [ ]:
spark = SparkSession.builder.appName("Structured").config("spark.sql.shuffle.partitions", 4).getOrCreate()

In [ ]:
raw = spark \
.readStream \
.format("kafka") \
.option("kafka.bootstrap.servers", "kafka:9092") \
.option("subscribe", "topic1") \
.load()

In [ ]:
words = raw.select(f.explode(f.split(raw.value, " ")).alias("value")).filter(f.col("value") != "")

SparkSession.conf.set("spark.sql.streaming.checkpointLocation", ...)

In [ ]:
query = words \
.writeStream \
.format("kafka") \
.option("kafka.bootstrap.servers", "kafka:9092") \
.option("topic", "topic2") \
.option("checkpointLocation", "chckpt") \
.start()

query.awaitTermination(60)
query.stop()

----

## Subskrypcja tematu - wiadomości ze schematem

- wymaga uruchomienia Kafki i utworzenia tematu `topic1` w sposób wskazany w pliku `kafka_commands` oraz uruchomienia skryptu `kafka_stream.py`

In [ ]:
from os import environ
environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.6 pyspark-shell'

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [ ]:
spark = SparkSession.builder.appName("Structured").config("spark.sql.shuffle.partitions", 4).getOrCreate()

In [ ]:
raw = spark \
.readStream \
.format("kafka") \
.option("kafka.bootstrap.servers", "kafka:9092") \
.option("subscribe", "topic1") \
.load()

In [ ]:
#schema = StructType([StructField("time", StringType()), StructField("number", StringType())])
#parsed = raw.select("key", f.from_json(raw.value.cast("string"), schema).alias("json"))

schema = StructType([StructField("time", StringType()), StructField("number", IntegerType()), 
                     StructField("string", StringType())])
parsed = raw.select(f.from_json(raw.value.cast("string"), schema).alias("json"))\
.select(f.col("json").getField("time").alias("time"), f.col("json").getField("number").alias("number"), 
        f.col("json").getField("string").alias("string"))

In [ ]:
query = parsed.writeStream.outputMode("append").format("console").start()
query.awaitTermination(60)
query.stop()